# MODULES

In [12]:
!pip install pathlib
!pip install matplotlib

In [13]:
!nvidia-smi

Mon Sep 29 10:45:08 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3060        Off | 00000000:01:00.0  On |                  N/A |
|  0%   50C    P8              17W / 170W |    261MiB / 12288MiB |     21%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [14]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

1.13.1+cu117
11.7
NVIDIA GeForce RTX 3060


In [15]:
import cv2
import os
import math
import time
import random
import pathlib
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import shutil


# GLOBAL VARIABLES

In [16]:
ROOT_DIR_IMAGES = '../Kaggle'

# IMAGES

In [17]:
def get_image_paths(root_dir, include='*', exclude=[]):
  paths = []
  
  for path in pathlib.Path(root_dir).glob(include): 
    paths.append(path)

  paths = sorted(paths)

  return paths

    
def base_filename_organization(imagPaths):
    baseFileNames = {}
    for imagPath in imagPaths:
        baseFileName = '_'.join(str(imagPath.stem).split('_')[:-1])
        if baseFileName not in baseFileNames:
            baseFileNames[baseFileName] = []
        baseFileNames[baseFileName].append(imagPath)

    # Sort lists numerically by the last suffix
    for key in baseFileNames:
        baseFileNames[key].sort(
            key=lambda p: int(p.stem.split('_')[-1])
        )

    return baseFileNames


def remove_last_element(baseFileNames):
    for baseFileName in baseFileNames:
        imagPaths = baseFileNames[baseFileName]
        length = len(imagPaths)
        baseFileNameLength = baseFileName + f'_{length-1}'
        imagPaths = [imagPath for imagPath in imagPaths if baseFileNameLength not in str(imagPath)]
        baseFileNames[baseFileName] = imagPaths
    return baseFileNames


def make_composite_images(images_dict: dict, final_width: int, final_height: int, save_dir: str):
    """
    Creates composite images from dict of images using OpenCV.

    Args:
        images_dict (dict): {key: [list of image paths]}
        final_width (int): width of final composite image
        final_height (int): height of final composite image
        save_dir (str): directory to save composite results
    """
    pathlib.Path(save_dir).mkdir(parents=True, exist_ok=True)

    for key, img_paths in images_dict.items():
        n = len(img_paths)
        grid_size = math.ceil(math.sqrt(n))  # square-ish grid
        cell_w = final_width // grid_size
        cell_h = final_height // grid_size

        # Black canvas
        composite = np.zeros((final_height, final_width, 3), dtype=np.uint8)

        for idx, img_path in enumerate(img_paths):
            try:
                img = cv2.imread(str(img_path))
                if img is None:
                    print(f"⚠️ Could not load {img_path}")
                    continue

                img = cv2.resize(img, (cell_w, cell_h), interpolation=cv2.INTER_AREA)

                row, col = divmod(idx, grid_size)
                y, x = row * cell_h, col * cell_w
                composite[y:y+cell_h, x:x+cell_w] = img
            except Exception as e:
                print(f"⚠️ Error with {img_path}: {e}")

        out_path = str(pathlib.Path(save_dir) / f"{key}.jpg")
        cv2.imwrite(out_path, composite)
        print(f"✅ Saved {out_path}")

In [18]:
imagsPaths = {}
imagsPaths['fall']    = get_image_paths( ROOT_DIR_IMAGES + '/Fall/Square_Image_Rotated')
imagsPaths['no_fall'] = get_image_paths( ROOT_DIR_IMAGES + '/No_Fall/Square_Image_Rotated')

print(f"[INFO] videos dict keys: {list(imagsPaths.keys())}")
print(f"[INFO] videos['fall'] has {len(imagsPaths['fall'])} entries")
print(f"[INFO] videos['no_fall'] has {len(imagsPaths['no_fall'])} entries")

[INFO] videos dict keys: ['fall', 'no_fall']
[INFO] videos['fall'] has 15680 entries
[INFO] videos['no_fall'] has 19110 entries


In [19]:
def save_images_by_prefix(fallByPrfix, base_save_dir):
    """
    Save images into folders organized by their prefix.

    Parameters:
        fallByPrfix (dict): Dictionary where keys are prefixes and values are lists of image paths.
        base_save_dir (str): Base directory where images will be saved in subfolders by prefix.
    """
    os.makedirs(base_save_dir, exist_ok=True)

    for prefix, paths in fallByPrfix.items():
        prefix_dir = os.path.join(base_save_dir, prefix)
        os.makedirs(prefix_dir, exist_ok=True)

        for img_path in paths:
            try:
                # Copy image to the prefix folder
                shutil.copy(img_path, prefix_dir)
            except Exception as e:
                print(f"[ERROR] Could not copy {img_path} to {prefix_dir}: {e}")

    print(f"[INFO] Images saved to {base_save_dir} organized by prefix.")

# FALL BY PREFIX

In [20]:
fallByPrfix = {}

fallByPrfix['20240912'] = []
fallByPrfix['20240913'] = []
fallByPrfix['20240914'] = []
fallByPrfix['20240915'] = []
fallByPrfix['20240916'] = []
fallByPrfix['20240917'] = []
fallByPrfix['20240918'] = []
fallByPrfix['20240919'] = []
fallByPrfix['C_D'] = []
fallByPrfix['C_M'] = []
fallByPrfix['C_N'] = []
fallByPrfix['S_D'] = []
fallByPrfix['S_F20'] = []
fallByPrfix['S_F19'] = []
fallByPrfix['S_F18'] = []
fallByPrfix['S_F17'] = []
fallByPrfix['S_F16'] = []
fallByPrfix['S_F15'] = []
fallByPrfix['S_F14'] = []
fallByPrfix['S_F13'] = []
fallByPrfix['S_F12'] = []
fallByPrfix['S_F11'] = []
fallByPrfix['S_F10'] = []
fallByPrfix['S_F9'] = []
fallByPrfix['S_F8'] = []
fallByPrfix['S_F7'] = []
fallByPrfix['S_F6'] = []
fallByPrfix['S_F5'] = []
fallByPrfix['S_F4'] = []
fallByPrfix['S_F3'] = []
fallByPrfix['S_F2'] = []
fallByPrfix['S_F1'] = []
fallByPrfix['S_M'] = []
fallByPrfix['S_N'] = []


print(len(imagsPaths['fall']))
imagsPathsFallNew = []
for en, path in enumerate(imagsPaths['fall']):
    print(en, path)
    stem = pathlib.Path(path).stem 
    prefixMatch = False
    for prefix in fallByPrfix:
        if prefix in stem:
            fallByPrfix[prefix].append(path)
            prefixMatch = True
            break
    if not prefixMatch:
        imagsPathsFallNew.append(path)
imagsPaths['fall'] = imagsPathsFallNew
print(len(imagsPaths['fall']))
print(sum([len(lst) for lst in fallByPrfix.values()]))

15680
0 ../Kaggle/Fall/Square_Image_Rotated/20240912_101331_1.jpg
1 ../Kaggle/Fall/Square_Image_Rotated/20240912_101331_2.jpg
2 ../Kaggle/Fall/Square_Image_Rotated/20240912_101331_3.jpg
3 ../Kaggle/Fall/Square_Image_Rotated/20240912_101331_4.jpg
4 ../Kaggle/Fall/Square_Image_Rotated/20240912_101331_5.jpg
5 ../Kaggle/Fall/Square_Image_Rotated/20240912_101427_1.jpg
6 ../Kaggle/Fall/Square_Image_Rotated/20240912_101427_2.jpg
7 ../Kaggle/Fall/Square_Image_Rotated/20240912_101427_3.jpg
8 ../Kaggle/Fall/Square_Image_Rotated/20240912_101427_4.jpg
9 ../Kaggle/Fall/Square_Image_Rotated/20240912_101427_5.jpg
10 ../Kaggle/Fall/Square_Image_Rotated/20240912_101520_1.jpg
11 ../Kaggle/Fall/Square_Image_Rotated/20240912_101520_2.jpg
12 ../Kaggle/Fall/Square_Image_Rotated/20240912_101520_3.jpg
13 ../Kaggle/Fall/Square_Image_Rotated/20240912_101520_4.jpg
14 ../Kaggle/Fall/Square_Image_Rotated/20240912_101520_5.jpg
15 ../Kaggle/Fall/Square_Image_Rotated/20240912_101626_1.jpg
16 ../Kaggle/Fall/Square_Ima

In [21]:
imagsPaths['fall'] = imagsPathsFallNew
print(len(imagsPaths['fall']))
print(sum([len(lst) for lst in fallByPrfix.values()]))

0
15680


In [22]:
save_images_by_prefix(fallByPrfix, ROOT_DIR_IMAGES + '/Fall/Square_Image_Rotated_Prefix')

[INFO] Images saved to ../Kaggle/Fall/Square_Image_Rotated_Prefix organized by prefix.


# NO FALL BY PREFIX

In [23]:
noFallByPrfix = {}

noFallByPrfix['B_D'] = []
noFallByPrfix['B_M'] = []
noFallByPrfix['B_N'] = []
noFallByPrfix['C0'] = []
noFallByPrfix['C_D'] = []
noFallByPrfix['C_M'] = []
noFallByPrfix['C_N'] = []
noFallByPrfix['L0'] = []
noFallByPrfix['R0'] = []
noFallByPrfix['S_D'] = []
noFallByPrfix['S_M'] = []
noFallByPrfix['S_N'] = []
noFallByPrfix['W0'] = []


print(len(imagsPaths['no_fall']))
imagsPathsNoFallNew = []

for en, path in enumerate(imagsPaths['no_fall']):
    print(en, path)
    stem = pathlib.Path(path).stem 
    prefixMatch = False
    for prefix in noFallByPrfix:
        if prefix in stem:
            noFallByPrfix[prefix].append(path)
            prefixMatch = True
            break
    if not prefixMatch:
        imagsPathsNoFallNew.append(path)
imagsPaths['no_fall'] = imagsPathsNoFallNew
print(len(imagsPaths['no_fall']))
print(sum([len(lst) for lst in noFallByPrfix.values()]))

19110
0 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0001_1.jpg
1 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0001_2.jpg
2 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0001_3.jpg
3 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0001_4.jpg
4 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0001_5.jpg
5 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0002_1.jpg
6 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0002_2.jpg
7 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0002_3.jpg
8 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0002_4.jpg
9 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0002_5.jpg
10 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0003_1.jpg
11 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0003_2.jpg
12 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0003_3.jpg
13 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0003_4.jpg
14 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0003_5.jpg
15 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0004_1.jpg
16 ../Kaggle/No_Fall/Square_Image_Rotated/B_D_0004_2.jpg
17 ../Kaggle/No_Fall/Square_Image_R

In [24]:
imagsPaths['no_fall'] = imagsPathsNoFallNew
print(len(imagsPaths['no_fall']))
print(sum([len(lst) for lst in noFallByPrfix.values()]))

0
19110


In [25]:
save_images_by_prefix(noFallByPrfix, ROOT_DIR_IMAGES + '/No_Fall/Square_Image_Rotated_Prefix')

[INFO] Images saved to ../Kaggle/No_Fall/Square_Image_Rotated_Prefix organized by prefix.
